# 4.6 Naive Bayes
**Chapter 4 — Classification Algorithms**

---

## The Analogy

Imagine Suresh, founder of a Kerala startup, gets 100 emails a day. After months of reading them, his brain has learned:
- Emails with "Win", "Free", "Prize" → almost always spam
- Emails with "Meeting", "Report", "Invoice" → almost always legitimate

He doesn't memorise every email. He just **counts patterns** and **uses probability**.

That's Naive Bayes. It:
1. Counts how often each word appears in spam vs ham during training
2. When a new email arrives, asks: *given these words, is this more likely spam or ham?*
3. Uses **Bayes' Theorem** to calculate that probability

---

## Why "Naive"?

It assumes every word is **independent** of every other word, given the class.

In reality, "Win" and "Free" appear together in spam all the time — they're correlated. Naive Bayes ignores this and treats each word separately.

Is this realistic? **No.** Does it still work? **Often yes** — because you don't need exact probabilities, just the right ranking.

---

## Bayes' Theorem (you know this from Chapter 1.3)

$$P(\text{Spam} | \text{words}) = \frac{P(\text{words} | \text{Spam}) \cdot P(\text{Spam})}{P(\text{words})}$$

Three pieces:
- **P(Spam)** — Prior: what fraction of all emails are spam?
- **P(words | Spam)** — Likelihood: how often do these words appear in spam?
- **P(words)** — Evidence: same for all classes, so we ignore it when comparing

We calculate this for both Spam and Ham, pick the winner.

---

## The Three Types

| Type | Data | What it measures |
|------|------|------------------|
| **Multinomial** | Text (word counts) | How often does word X appear in class Y? |
| **Bernoulli** | Text (word present/absent) | Does word X appear at all in class Y? |
| **Gaussian** | Continuous numbers | Is this value likely under class Y's bell curve? |

For spam classification → **Multinomial** (word counts capture intensity of spam signals)

---

## Two Important Fixes

### Fix 1 — Laplace Smoothing (alpha=1)
If a word never appeared in training spam emails, P(word|Spam) = 0.
One zero kills the entire product: 0.13 × 0.04 × 0 × 0.07 = **0**. Wrong.

Fix: add 1 to every word count so no probability is ever exactly zero.
$$P(\text{word} | \text{class}) = \frac{\text{count} + 1}{\text{total words in class} + \text{vocabulary size}}$$

### Fix 2 — Log Trick (automatic in sklearn)
Multiplying hundreds of tiny probabilities → number too small for a computer to store (underflow).

Fix: use logarithms. Multiplication becomes addition:
$$\log(A \times B \times C) = \log(A) + \log(B) + \log(C)$$

sklearn does this automatically. You never see it, but it's always happening.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, roc_curve
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

## Part 1 — Naive Bayes From Scratch

Before using sklearn, let's build a tiny Multinomial Naive Bayes by hand.
This will make every sklearn parameter make sense later.

We'll use 6 toy messages — 3 spam, 3 ham.

In [ ]:
# --- TOY DATASET ---
# 6 messages: 3 spam, 3 ham
messages = [
    "win free prize now",       # spam
    "free offer claim prize",   # spam
    "win cash reward free",     # spam
    "meeting at office today",  # ham
    "report due tomorrow",      # ham
    "office lunch at noon",     # ham
]
labels = ["spam", "spam", "spam", "ham", "ham", "ham"]

# --- STEP 1: BUILD VOCABULARY ---
# Split every message into words and collect unique ones
# This is what CountVectorizer does internally
all_words = set()
for msg in messages:
    for word in msg.split():
        all_words.add(word)

vocab = sorted(all_words)  # sorted for consistency
print(f"Vocabulary ({len(vocab)} words): {vocab}")

# --- STEP 2: COUNT WORDS PER CLASS ---
# For each class, count how many times each word appears ACROSS ALL messages of that class
spam_counts = {word: 0 for word in vocab}
ham_counts  = {word: 0 for word in vocab}

for msg, label in zip(messages, labels):
    for word in msg.split():
        if label == "spam":
            spam_counts[word] += 1
        else:
            ham_counts[word] += 1

print("\nWord counts in SPAM:", {k: v for k, v in spam_counts.items() if v > 0})
print("Word counts in HAM: ", {k: v for k, v in ham_counts.items()  if v > 0})

In [ ]:
# --- STEP 3: COMPUTE PROBABILITIES WITH LAPLACE SMOOTHING ---
# Without smoothing: any unseen word → P = 0 → entire product = 0 (wrong!)
# With smoothing (alpha=1): add 1 to every count so nothing is ever zero

alpha = 1  # Laplace smoothing — same as sklearn's default
V = len(vocab)  # vocabulary size — needed in the denominator

total_spam_words = sum(spam_counts.values())  # total words across all spam messages
total_ham_words  = sum(ham_counts.values())   # total words across all ham messages

# P(word | spam) = (count_in_spam + 1) / (total_spam_words + V)
# The +V in denominator ensures all probabilities still sum to 1 after adding 1 to each
spam_probs = {word: (spam_counts[word] + alpha) / (total_spam_words + V) for word in vocab}
ham_probs  = {word: (ham_counts[word]  + alpha) / (total_ham_words  + V) for word in vocab}

# Prior probabilities — what fraction of messages are spam vs ham?
n_spam = labels.count("spam")
n_ham  = labels.count("ham")
n_total = len(labels)

prior_spam = n_spam / n_total  # P(spam)
prior_ham  = n_ham  / n_total  # P(ham)

print(f"Prior P(spam) = {prior_spam:.2f}")
print(f"Prior P(ham)  = {prior_ham:.2f}")
print(f"\nP('free' | spam) = {spam_probs['free']:.4f}")
print(f"P('free' | ham)  = {ham_probs['free']:.4f}")
print(f"\n'free' is {spam_probs['free']/ham_probs['free']:.1f}x more likely in spam than ham")

In [ ]:
# --- STEP 4: PREDICT A NEW MESSAGE ---
# We use LOG probabilities to avoid underflow (multiplying tiny numbers → 0)
# log(A × B × C) = log(A) + log(B) + log(C)  — addition, not multiplication

def predict_naive_bayes(message):
    words = message.lower().split()
    
    # Start with log of prior probability
    log_prob_spam = np.log(prior_spam)
    log_prob_ham  = np.log(prior_ham)
    
    for word in words:
        if word in vocab:
            # Add log probability for each word (instead of multiplying raw probabilities)
            log_prob_spam += np.log(spam_probs[word])
            log_prob_ham  += np.log(ham_probs[word])
        # If word not in vocab: skip (Laplace handles known words; unknown words we ignore)
    
    print(f"Message: '{message}'")
    print(f"  log P(spam | message) = {log_prob_spam:.4f}")
    print(f"  log P(ham  | message) = {log_prob_ham:.4f}")
    print(f"  Prediction: {'SPAM' if log_prob_spam > log_prob_ham else 'HAM'}")
    print()

# Test on new messages
predict_naive_bayes("win free prize")     # should be spam
predict_naive_bayes("office meeting")     # should be ham
predict_naive_bayes("free office lunch")  # ambiguous — which wins?

## Part 2 — Real Dataset: SMS Spam Collection

5,572 real SMS messages, labelled spam or ham.
We'll download it directly from UCI.

In [ ]:
# Load the SMS Spam Collection dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nSpam %: {df['label'].value_counts(normalize=True)['spam']*100:.1f}%")
print(f"\nSample spam messages:")
print(df[df['label']=='spam']['message'].head(3).to_string())
print(f"\nSample ham messages:")
print(df[df['label']=='ham']['message'].head(3).to_string())

In [ ]:
# --- VISUALISE CLASS DISTRIBUTION ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('SMS Spam Dataset Overview', fontsize=14, fontweight='bold')

# Class balance
counts = df['label'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, count + 30, str(count), ha='center', fontweight='bold')

# Message length distribution
df['msg_length'] = df['message'].apply(len)
for label, color in zip(['ham', 'spam'], ['#2ecc71', '#e74c3c']):
    subset = df[df['label'] == label]['msg_length']
    axes[1].hist(subset, bins=50, alpha=0.6, color=color, label=label)
axes[1].set_title('Message Length Distribution')
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Average spam length:  {df[df['label']=='spam']['msg_length'].mean():.0f} characters")
print(f"Average ham length:   {df[df['label']=='ham']['msg_length'].mean():.0f} characters")
print("\nNote: Spam messages tend to be LONGER — more words = more 'urgent' claims")

## Part 3 — Building the Classifier

**Pipeline:**
```
Raw text → CountVectorizer (text → word counts) → MultinomialNB → prediction
```

We use Pipeline to prevent data leakage — CountVectorizer must fit on training data only.

In [ ]:
# --- PREPARE DATA ---
X = df['message']           # raw text messages
y = (df['label'] == 'spam').astype(int)  # 1 = spam, 0 = ham

# Split BEFORE any text processing — CountVectorizer must only see training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # preserve spam/ham ratio in both splits
)

print(f"Training set:  {len(X_train)} messages")
print(f"Test set:      {len(X_test)} messages")
print(f"Spam in train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Spam in test:  {y_test.sum()} ({y_test.mean()*100:.1f}%)")

In [ ]:
# --- BUILD PIPELINE ---
# WHY Pipeline? CountVectorizer must learn vocabulary from TRAINING data only.
# If we fit it on all data first, test words influence the vocabulary → data leakage.

pipeline = Pipeline([
    ('vectorizer', CountVectorizer(
        stop_words='english',  # remove 'the', 'is', 'at' etc — they don't help classify spam
        lowercase=True,        # 'FREE' and 'free' should be treated as same word
        max_features=5000      # keep only top 5000 most frequent words — reduces noise
    )),
    ('classifier', MultinomialNB(
        alpha=1.0  # Laplace smoothing — prevents any word from having P=0
    ))
])

# Train — CountVectorizer learns vocabulary, NB counts word frequencies per class
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]  # probability of SPAM (class 1)

print("Model trained successfully!")
print(f"\nVocabulary size: {len(pipeline.named_steps['vectorizer'].vocabulary_)} words")

In [ ]:
# --- EVALUATE ---
print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {auc:.4f}")

# Cross-validation to confirm stability
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='f1')
print(f"\n5-Fold CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# --- VISUALISE: CONFUSION MATRIX + ROC CURVE ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Naive Bayes — Model Evaluation', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Ham', 'Spam'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## Part 4 — What Did The Model Learn?

Naive Bayes is one of the most **interpretable** models — we can directly inspect which words it associates most strongly with spam vs ham.

In [ ]:
# --- MOST INFORMATIVE WORDS ---
# The classifier stores log probabilities for each word per class
# Higher log prob = word appears more often in that class

nb_model   = pipeline.named_steps['classifier']
vectorizer = pipeline.named_steps['vectorizer']
feature_names = vectorizer.get_feature_names_out()

# Log probability difference: high = strongly spam, low = strongly ham
# nb_model.feature_log_prob_[1] = log P(word | spam)
# nb_model.feature_log_prob_[0] = log P(word | ham)
log_prob_diff = nb_model.feature_log_prob_[1] - nb_model.feature_log_prob_[0]

n_top = 15
top_spam_idx = np.argsort(log_prob_diff)[-n_top:][::-1]  # highest diff = most spam-like
top_ham_idx  = np.argsort(log_prob_diff)[:n_top]          # lowest diff = most ham-like

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Most Informative Words per Class', fontsize=14, fontweight='bold')

# Spam words
spam_words  = [feature_names[i] for i in top_spam_idx]
spam_scores = [log_prob_diff[i] for i in top_spam_idx]
axes[0].barh(spam_words[::-1], spam_scores[::-1], color='#e74c3c', edgecolor='black', lw=0.5)
axes[0].set_title('Top SPAM words', color='#e74c3c')
axes[0].set_xlabel('Log probability difference (spam − ham)')

# Ham words
ham_words  = [feature_names[i] for i in top_ham_idx]
ham_scores = [log_prob_diff[i] for i in top_ham_idx]
axes[1].barh(ham_words[::-1], ham_scores[::-1], color='#2ecc71', edgecolor='black', lw=0.5)
axes[1].set_title('Top HAM words', color='#2ecc71')
axes[1].set_xlabel('Log probability difference (spam − ham)')

plt.tight_layout()
plt.show()

## Part 5 — Comparing All Three NB Types

Let's see how Multinomial, Bernoulli, and Gaussian compare on this dataset.

In [ ]:
# --- COMPARE ALL THREE NB TYPES ---
results = {}

# 1. Multinomial NB (word counts)
mnb_pipeline = Pipeline([
    ('vec', CountVectorizer(stop_words='english', lowercase=True, max_features=5000)),
    ('clf', MultinomialNB(alpha=1.0))
])

# 2. Bernoulli NB (word presence/absence — binary)
bnb_pipeline = Pipeline([
    ('vec', CountVectorizer(stop_words='english', lowercase=True,
                            max_features=5000, binary=True)),  # binary=True → 0/1 only
    ('clf', BernoulliNB(alpha=1.0))
])

# 3. Gaussian NB — needs dense numerical features
# We'll use TF-IDF as features and convert to dense array
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=500)  # smaller for memory
X_tfidf_train = tfidf.fit_transform(X_train).toarray()  # must be dense for Gaussian
X_tfidf_test  = tfidf.transform(X_test).toarray()

gnb = GaussianNB()
gnb.fit(X_tfidf_train, y_train)

# Evaluate all three
for name, pipeline_obj, X_tr, X_te in [
    ('Multinomial NB', mnb_pipeline, X_train, X_test),
    ('Bernoulli NB',   bnb_pipeline, X_train, X_test),
]:
    pipeline_obj.fit(X_tr, y_train)
    preds = pipeline_obj.predict(X_te)
    probs = pipeline_obj.predict_proba(X_te)[:, 1]
    from sklearn.metrics import f1_score, precision_score, recall_score
    results[name] = {
        'F1':       f1_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall':   recall_score(y_test, preds),
        'AUC':      roc_auc_score(y_test, probs)
    }

# Gaussian
from sklearn.metrics import f1_score, precision_score, recall_score
gnb_preds = gnb.predict(X_tfidf_test)
gnb_probs = gnb.predict_proba(X_tfidf_test)[:, 1]
results['Gaussian NB'] = {
    'F1':       f1_score(y_test, gnb_preds),
    'Precision': precision_score(y_test, gnb_preds),
    'Recall':   recall_score(y_test, gnb_preds),
    'AUC':      roc_auc_score(y_test, gnb_probs)
}

# Display results
results_df = pd.DataFrame(results).T.round(4)
print(results_df)
print("\nNote: Multinomial NB is best for text — it's designed for word counts.")
print("Gaussian NB is worst here — it's designed for continuous data, not text.")

In [ ]:
# --- VISUALISE COMPARISON ---
fig, ax = plt.subplots(figsize=(10, 5))

metrics = ['F1', 'Precision', 'Recall', 'AUC']
x = np.arange(len(metrics))
width = 0.25
colors = ['#3498db', '#e67e22', '#9b59b6']

for i, (model_name, color) in enumerate(zip(results.keys(), colors)):
    values = [results[model_name][m] for m in metrics]
    bars = ax.bar(x + i * width, values, width, label=model_name, color=color,
                  alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Naive Bayes Type Comparison — SMS Spam')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 6 — Try Your Own Messages

In [ ]:
# --- PREDICT ON CUSTOM MESSAGES ---
# Try any message you want!

custom_messages = [
    "Congratulations! You have won a free iPhone. Call now to claim your prize!",
    "Hey, are you coming for the team lunch at 1pm?",
    "URGENT: Your bank account has been compromised. Click here immediately.",
    "The project report is due by Friday. Please review the attached document.",
    "Win Rs 1 lakh today! Free offer, limited time only. Reply YES to claim."
]

predictions = pipeline.predict(custom_messages)
probabilities = pipeline.predict_proba(custom_messages)[:, 1]

print(f"{'Message':<60} {'Pred':<8} {'Spam Prob':<10}")
print("-" * 80)
for msg, pred, prob in zip(custom_messages, predictions, probabilities):
    label = "SPAM" if pred == 1 else "HAM"
    print(f"{msg[:58]:<60} {label:<8} {prob:.4f}")

## Summary Table

| Concept | What it means |
|---------|---------------|
| **Prior P(class)** | How common is this class in training data? |
| **Likelihood P(word\|class)** | How often does this word appear in this class? |
| **Laplace smoothing (alpha)** | Prevents P=0 for unseen words; add 1 to all counts |
| **Log trick** | Prevents underflow; multiply→add in log space |
| **CountVectorizer** | Converts text to word count vectors |
| **Multinomial NB** | Best for text with word counts |
| **Bernoulli NB** | Best for short text, binary presence/absence |
| **Gaussian NB** | Best for continuous numerical features |
| **Pipeline** | Ensures vectorizer fits on train only — prevents leakage |

---

## When to Use Naive Bayes

| Use NB when... | Avoid NB when... |
|----------------|------------------|
| Data is text (spam, news, sentiment) | Features are highly correlated (NB assumes independence) |
| Dataset is very large (NB is fast) | You need probability calibration (NB probabilities are poorly calibrated) |
| You need a quick baseline | Feature interactions matter for prediction |
| Training data is limited | Numerical features with complex distributions |

---

## Practice Task

You work at a hospital in Kochi. Doctors receive patient emails and want to flag **urgent** ones automatically.

A small labelled dataset:
- urgent: "chest pain severe", "cannot breathe help", "bleeding wont stop urgent"
- routine: "appointment next week", "prescription refill request", "lab results query"

**Task 1:** Using the from-scratch approach from Part 1, build a Naive Bayes classifier for this dataset and predict whether "severe chest pain please help" is urgent or routine.

In [ ]:
# TASK 1: From-scratch Naive Bayes for hospital email triage
# Hint: follow the same 4-step pattern from Part 1

hospital_messages = [
    "chest pain severe",
    "cannot breathe help",
    "bleeding wont stop urgent",
    "appointment next week",
    "prescription refill request",
    "lab results query"
]
hospital_labels = ["urgent", "urgent", "urgent", "routine", "routine", "routine"]

# YOUR CODE HERE
# Step 1: Build vocabulary

# Step 2: Count words per class

# Step 3: Compute probabilities with Laplace smoothing

# Step 4: Predict "severe chest pain please help"

In [ ]:
# TASK 2: Now use sklearn Pipeline to do the same thing
# Train on hospital_messages, predict on ["severe chest pain please help", "schedule appointment"]

# YOUR CODE HERE